# Baseline Regression Models

Bu notebook'ta SELFIES tabanlı node ve linker girdileri kullanılarak
density, PLD ve LCD hedefleri için ayrı regresyon modelleri
eğitilmektedir.

Bütün algoritmalar aynı train, validation ve test bölünmesini
kullanmaktadır. Test seti yalnızca son değerlendirme aşamasında
kullanılacaktır.

In [1]:
from pathlib import Path
import time

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor


PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "forward_model_selfies.jsonl"
)

RESULTS_DIR = PROJECT_ROOT / "results" / "tables"
MODELS_DIR = PROJECT_ROOT / "results" / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_json(
    DATA_PATH,
    lines=True,
)

print("Veri seti boyutu:", df.shape)

display(df.head())

Veri seti boyutu: (17348, 24)


,qmof_id,smiles_nodes,smiles_linkers,point_group,topology,density,pld,lcd,topology_missing,node_smiles_list,...,linker_smiles_list,linker_selfies_list,linker_tokens,linker_relaxed_count,linker_failed_count,node_token_length,linker_token_length,selfies_encoding_ok,node_token_text,linker_token_text
0,qmof-8a95c27,"['O', '[Ba]', '[Cu]']",['[O-]C=O'],-1,__MISSING_TOPOLOGY__,2.763246,0.68822,1.35480,1,"[O, [Ba], [Cu]]",...,[[O-]C=O],[[O-1][C][=O]],"[[O-1], [C], [=O]]",0,0,5,3,True,[O] <MOL_SEP> [Ba] <MOL_SEP> [Cu],[O-1] [C] [=O]
1,qmof-830ed1c,['[Co]'],['[O-]C(=O)c1ccncc1'],m,rtl,1.557644,2.36128,4.21176,0,[[Co]],...,[[O-]C(=O)c1ccncc1],[[O-1][C][=Branch1][C][=O][C][=C][C][=N][C][=C...,"[[O-1], [C], [=Branch1], [C], [=O], [C], [=C],...",0,0,1,13,True,[Co],[O-1] [C] [=Branch1] [C] [=O] [C] [=C] [C] [=N...
2,qmof-5bd4a24,['[Co]'],['[O-]C(=O)c1ccncc1'],2/m,rtl,1.616139,2.14542,3.27957,0,[[Co]],...,[[O-]C(=O)c1ccncc1],[[O-1][C][=Branch1][C][=O][C][=C][C][=N][C][=C...,"[[O-1], [C], [=Branch1], [C], [=O], [C], [=C],...",0,0,1,13,True,[Co],[O-1] [C] [=Branch1] [C] [=O] [C] [=C] [C] [=N...
3,qmof-644aab4,['[Zn][Zn]'],"['[O-]C(=O)c1cccc(c1)c1nccs1', 'n1ccc(cc1)c1cc...",-1,__MISSING_TOPOLOGY__,1.596537,1.33452,2.03948,1,[[Zn][Zn]],...,"[[O-]C(=O)c1cccc(c1)c1nccs1, n1ccc(cc1)c1ccncc1]",[[O-1][C][=Branch1][C][=O][C][=C][C][=C][C][=B...,"[[O-1], [C], [=Branch1], [C], [=O], [C], [=C],...",0,0,2,41,True,[Zn] [Zn],[O-1] [C] [=Branch1] [C] [=O] [C] [=C] [C] [=C...
4,qmof-eaa4957,"['[OH2][Ag][Ag][Ag][Ag][OH2]', '[OH2][Ag][Ag][...","['[O-]C(=O)C1=NN=C([CH]1)C(=O)[O-]', '[O-]C(=O...",-1,__MISSING_TOPOLOGY__,3.421198,0.78190,2.42132,1,"[[OH2][Ag][Ag][Ag][Ag][OH2], [OH2][Ag][Ag][OH2]]",...,"[[O-]C(=O)C1=NN=C([CH]1)C(=O)[O-], [O-]C(=O)C1...",[[O-1][C][=Branch1][C][=O][C][=N][N][=C][Branc...,"[[O-1], [C], [=Branch1], [C], [=O], [C], [=N],...",0,0,11,39,True,[OH2] [Ag] [Ag] [Ag] [Ag] [OH2] <MOL_SEP> [OH2...,[O-1] [C] [=Branch1] [C] [=O] [C] [=N] [N] [=C...


In [2]:
feature_columns = [
    "node_token_text",
    "linker_token_text",
    "point_group",
    "topology",
    "topology_missing",
]

target_columns = [
    "density",
    "pld",
    "lcd",
]

X = df[feature_columns].copy()
y = df[target_columns].copy()

print("Girdi boyutu :", X.shape)
print("Hedef boyutu :", y.shape)

Girdi boyutu : (17348, 5)
Hedef boyutu : (17348, 3)


In [3]:
all_indices = np.arange(len(df))

train_indices, temporary_indices = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=42,
    shuffle=True,
)

validation_indices, test_indices = train_test_split(
    temporary_indices,
    test_size=0.50,
    random_state=42,
    shuffle=True,
)

train_df = df.iloc[train_indices].copy()
validation_df = df.iloc[validation_indices].copy()
test_df = df.iloc[test_indices].copy()

print("Train      :", train_df.shape)
print("Validation :", validation_df.shape)
print("Test       :", test_df.shape)

Train      : (12143, 24)
Validation : (2602, 24)
Test       : (2603, 24)


In [4]:
split_assignments = pd.DataFrame({
    "qmof_id": df["qmof_id"],
    "split": "unassigned",
})

split_assignments.loc[train_indices, "split"] = "train"
split_assignments.loc[validation_indices, "split"] = "validation"
split_assignments.loc[test_indices, "split"] = "test"

split_assignments.to_csv(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "split_assignments.csv",
    index=False,
)

display(
    split_assignments["split"]
    .value_counts()
    .to_frame()
)

,count
split,
train,12143
test,2603
validation,2602


In [5]:
node_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
)

linker_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
)

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "node_tfidf",
            node_vectorizer,
            "node_token_text",
        ),
        (
            "linker_tfidf",
            linker_vectorizer,
            "linker_token_text",
        ),
        (
            "categorical",
            categorical_encoder,
            [
                "point_group",
                "topology",
            ],
        ),
        (
            "missing_indicator",
            "passthrough",
            [
                "topology_missing",
            ],
        ),
    ],
    remainder="drop",
)

In [6]:
model_definitions = {
    "dummy_mean": DummyRegressor(
        strategy="mean",
    ),

    "linear_regression": LinearRegression(),

    "ridge": Ridge(
        alpha=1.0,
        solver="lsqr",
    ),

    "decision_tree": DecisionTreeRegressor(
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
    ),

    "random_forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
}

In [7]:
def calculate_regression_metrics(
    y_true,
    y_pred,
):
    mae = mean_absolute_error(
        y_true,
        y_pred,
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred,
        )
    )

    r2 = r2_score(
        y_true,
        y_pred,
    )

    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
    }

In [8]:
X_train = train_df[feature_columns]
X_validation = validation_df[feature_columns]

results = []
trained_models = {}

for target in target_columns:
    y_train = train_df[target]
    y_validation = validation_df[target]

    print("=" * 70)
    print("Hedef:", target)

    for model_name, estimator in model_definitions.items():
        print(f"Eğitiliyor: {model_name}")

        pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    clone(preprocessor),
                ),
                (
                    "model",
                    clone(estimator),
                ),
            ]
        )

        start_time = time.perf_counter()

        pipeline.fit(
            X_train,
            y_train,
        )

        training_seconds = (
            time.perf_counter() - start_time
        )

        validation_predictions = pipeline.predict(
            X_validation
        )

        metrics = calculate_regression_metrics(
            y_validation,
            validation_predictions,
        )

        result = {
            "target": target,
            "model": model_name,
            "validation_mae": metrics["mae"],
            "validation_rmse": metrics["rmse"],
            "validation_r2": metrics["r2"],
            "training_seconds": training_seconds,
        }

        results.append(result)

        trained_models[
            (target, model_name)
        ] = pipeline

        print(
            f"MAE={metrics['mae']:.4f} | "
            f"RMSE={metrics['rmse']:.4f} | "
            f"R²={metrics['r2']:.4f} | "
            f"Süre={training_seconds:.2f} sn"
        )

Hedef: density
Eğitiliyor: dummy_mean
MAE=0.4788 | RMSE=0.6272 | R²=-0.0000 | Süre=0.35 sn
Eğitiliyor: linear_regression
MAE=0.2341 | RMSE=0.3718 | R²=0.6486 | Süre=2.87 sn
Eğitiliyor: ridge
MAE=0.2140 | RMSE=0.2851 | R²=0.7933 | Süre=0.41 sn
Eğitiliyor: decision_tree
MAE=0.2526 | RMSE=0.3633 | R²=0.6645 | Süre=1.83 sn
Eğitiliyor: random_forest
MAE=0.1929 | RMSE=0.2771 | R²=0.8048 | Süre=27.14 sn
Hedef: pld
Eğitiliyor: dummy_mean
MAE=2.5001 | RMSE=3.5887 | R²=-0.0000 | Süre=0.33 sn
Eğitiliyor: linear_regression
MAE=1.2804 | RMSE=2.0499 | R²=0.6737 | Süre=2.53 sn
Eğitiliyor: ridge
MAE=1.1556 | RMSE=1.7281 | R²=0.7681 | Süre=0.39 sn
Eğitiliyor: decision_tree
MAE=1.0998 | RMSE=1.9791 | R²=0.6959 | Süre=1.37 sn
Eğitiliyor: random_forest
MAE=0.8711 | RMSE=1.4841 | R²=0.8290 | Süre=31.68 sn
Hedef: lcd
Eğitiliyor: dummy_mean
MAE=2.9996 | RMSE=4.1991 | R²=-0.0000 | Süre=0.32 sn
Eğitiliyor: linear_regression
MAE=1.4452 | RMSE=2.2801 | R²=0.7051 | Süre=2.70 sn
Eğitiliyor: ridge
MAE=1.3081 | RMSE

In [9]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=[
        "target",
        "validation_rmse",
    ],
    ascending=[
        True,
        True,
    ],
)

display(results_df)

,target,model,validation_mae,validation_rmse,validation_r2,training_seconds
4,density,random_forest,0.192866,0.277099,8.048304e-01,27.139327
2,density,ridge,0.213969,0.285143,7.933341e-01,0.408992
3,density,decision_tree,0.252637,0.363313,6.644896e-01,1.830694
1,density,linear_regression,0.234112,0.371840,6.485569e-01,2.865158
0,density,dummy_mean,0.478839,0.627243,-3.430664e-05,0.354947
14,lcd,random_forest,0.976751,1.584069,8.576859e-01,28.924091
12,lcd,ridge,1.308081,1.892505,7.968700e-01,0.380630
13,lcd,decision_tree,1.228515,2.078748,7.549223e-01,1.392262
11,lcd,linear_regression,1.445179,2.280090,7.051482e-01,2.703882
10,lcd,dummy_mean,2.999600,4.199120,-3.775458e-05,0.318410
